<a href="https://colab.research.google.com/github/SaharWajahat/gmx_MMPBSA-Analysis-Template-GB-PB-Entropy-Decomp-Heatmap/blob/main/gmx_MMPBSA_Protein_LIG_GB_PB_Entropy_Decomp_Heatmap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Bismillah-ir-Rahman-ir-Raheem

## Reproducible gmx_MMPBSA + Entropy + Per-Residue Pipeline

**Author:** Sahar Wajahat
**Date:** November 2026
**License:** MIT — Free for academic/non-commercial use
**Contact:** saharwajahatfuuast@gmail.com
**ORCID:** [0009-0001-9005-5448] (https://orcid.org/0009-0001-9005-5448)

---

### Citation
If you use this software in your research, please cite:

> Wajahat, S. (2026). *gmx_MMPBSA-Analysis-Template-GB-PB-Entropy-Decomp-Heatmap: Reproducible gmx_MMPBSA Binding Free Energy Workflow* (Version 1.0.0) [Computer software]. https://github.com/SaharWajahat/gmx_MMPBSA-Analysis-Template-GB-PB-Entropy-Decomp-Heatmap

---
### link
https://github.com/SaharWajahat/gmx_MMPBSA-Analysis-Template-GB-PB-Entropy-Decomp-Heatmap

### 1. Purpose
A complete, end-to-end Colab notebook for **MM/GBSA & MM/PBSA** binding free energy analysis from GROMACS MD trajectories. Designed for reproducibility, transparency, and publication-ready plots. Focus on LIG-Protein systems.

### 2. What This Pipeline Does
1.  **Parse gmx_MMPBSA outputs:** `FINAL_DECOMP_last10ns_lig_23.csv` + `FINAL_DECOMP_last10ns_lig_23.dat` + `FINAL_RESULTS_last10ns_lig_23.csv` + `FINAL_RESULTS_last10ns_lig_23.dat`  (GB/PB)
2.  **Incorporate NMODE Entropy:** Correct binding energy with -TΔS for realistic ΔG
3.  **GB vs PB Comparison:** Side-by-side energy comparison (LIG_23 - GB vs PB)
4.  **Per-Residue Decomposition:** Identify key hotspot residues driving binding
5.  **Generate Publication Plots:** Heatmaps, bar plots, energy vs time - all 300 dpi
6.  **Export all outputs:** `.png` figures for thesis/paper

### 3. How To Use This Notebook
1.  **Upload files:** `FINAL_DECOMP_last10ns_lig_23.csv` + `FINAL_DECOMP_last10ns_lig_23.dat` + `FINAL_RESULTS_last10ns_lig_23.csv` + `FINAL_RESULTS_last10ns_lig_23.dat`
2.  **Set LIG name:** Edit `LIG_23` variable in Section 2 if your ligand is different
3.  **Run cells one by one** from top to bottom
4.  **Outputs:** figures: bindig free energies, entropy, per-residue decomposition, heatmaps

### 4. Requirements
- **Platform:** Google Colab - Free. No installation needed. Needs `matplotlib`, `numpy` , `seaborn`, `pandas`
- **Data:** Your own gmx_MMPBSA output files from GROMACS trajectory

### 5. Key Validation Feature
Unlike most tutorials that only show GB, this notebook includes **Section 5: GB vs PB + Entropy Validation.** Reviewers demand both GB/PB comparison and entropy correction. This pipeline generates that proof automatically, plus per-residue validation via heatmap.

### 6. Disclaimer
This is an academic tool based on gmx_MMPBSA (Valdés-Tresanco et al., JCTC 2021). Results depend on simulation length, force field, and entropy method. Not for clinical use without further testing.

---
*Built to make MMPBSA analysis reproducible for everyone. Alhamdullilah.*

In [ ]:
# For Energy and Entropy
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({'font.family':'sans-serif','font.size':7.5})

BG = '#EAEAF2'

dat_file = 'FINAL_RESULTS_last10ns_lig_23.dat'

def parse_mmpbsa_dat(dat_path, err_col='SD'):
    with open(dat_path, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()
    col_map = {'Average': 1, 'SD(Prop.)': 2, 'SD': 3, 'SEM(Prop.)': 4, 'SEM': 5}
    err_idx = col_map[err_col]
    result = {'GB': {}, 'PB': {}}
    current_method = None
    for i, line in enumerate(lines):
        if 'GENERALIZED BORN:' in line:
            current_method = 'GB'
        elif 'POISSON BOLTZMANN:' in line:
            current_method = 'PB'
        elif 'Delta (Complex - Receptor - Ligand):' in line and current_method:
            j = i + 3
            found_data = False
            while j < len(lines):
                l = lines[j].strip()
                if l.startswith('---'):
                    if found_data:
                        break
                    j += 1
                    continue
                if not l:
                    j += 1
                    continue
                parts = l.split()
                comp_orig = parts[0]
                if comp_orig.startswith('Δ'):
                    try:
                        avg = float(parts[1])
                        err = float(parts[err_idx])
                        # strip Δ for plotting
                        comp = comp_orig.replace('Δ','')
                        result[current_method][comp] = (avg, err)
                        result[current_method][comp_orig] = (avg, err)
                        found_data = True
                    except:
                        pass
                j += 1
    return result

# 1. Parse
vals = parse_mmpbsa_dat(dat_file, err_col='SD')
gb = vals['GB']
pb = vals['PB']

print("GB DELTA:")
for k, v in gb.items(): print(k, v)
print("\nPB DELTA:")
for k, v in pb.items(): print(k, v)

# 2. Plot
def plot_fig6(data, title, solv_keys, save):
    fig, axes = plt.subplots(1,3, figsize=(4.5,3.2))
    fig.suptitle(title, fontsize=11, fontweight='bold', x=0.02, ha='left')
    groups = [('GGAS',['VDWAALS','EEL']), ('GSOLV',solv_keys), ('TOTAL',['GGAS','GSOLV','TOTAL'])]
    colors = {'VDWAALS':'#E78CA2','EEL':'#C2A353','EGB':'#6AA84F','ESURF':'#7FC8BC',
          'EPB':'#6AA84F','ENPOLAR':'#7FC8BC','GGAS':'#4DB6AC','GSOLV':'#8A9AD5','TOTAL':'#E78CA2'}
    for ax, (gtitle, keys) in zip(axes, groups):
        ax.set_facecolor(BG)
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.set_title(gtitle, fontsize=10, color='Black')
        ax.xaxis.grid(False)
        vals = [data[k][0] for k in keys]
        errs = [data[k][1] for k in keys]
        bars = ax.bar(keys, vals, yerr=errs, capsize=3,
               color=[colors[k] for k in keys], edgecolor='white')
        # puts values inside bars
        #ax.bar_label(bars, fmt='%.2f', fontsize=7, label_type='center', color='black', fontweight='bold')
        ax.axhline(0, color='black', linewidth=0.6)
        ax.set_ylim(55, -82)
        plt.setp(ax.get_xticklabels(), rotation=30, ha='right', fontsize=8, color='Black')
        ax.yaxis.grid(True, color='white', linewidth=0.8)
        ax.set_axisbelow(True)
        ax.tick_params(axis='x', length=0) #line to remove x axis ticks
        ax.tick_params(axis='y', labelsize=8, labelcolor='Black')

    for ax in axes[1:]:
        ax.tick_params(axis='y', labelleft=False, left=False)

    plt.tight_layout(rect=[0,0,1,0.88])
    fig.subplots_adjust(wspace=0.04)
    axes[0].set_ylabel('Energy (kcal/mol)', fontsize=8)
    plt.savefig(save, dpi=300, bbox_inches='tight')
    print(f"Saved: {save}")
    plt.show()

plot_fig6(gb, 'A. lig_23 GB', ['EGB','ESURF'], 'fig1_gb_dat_Energy.png')
plot_fig6(pb, 'B. lig_23 PB', ['EPB','ENPOLAR'], 'fig2_pb_dat_Energy.png')

# --- Parse ---
dH_gb, dH_gb_std = vals['GB']['ΔTOTAL']
dH_pb, dH_pb_std = vals['PB']['ΔTOTAL']

# -TΔS from your NMODE block (same for GB and PB)
TdS_mean, TdS_std = 21.63, 3.50

# ΔG = ΔH + (-TΔS)
dG_gb = dH_gb + TdS_mean
dG_pb = dH_pb + TdS_mean
dG_gb_std = np.sqrt(dH_gb_std**2 + TdS_std**2)
dG_pb_std = np.sqrt(dH_pb_std**2 + TdS_std**2)

print(f"GB: ΔH={dH_gb:.2f}±{dH_gb_std:.2f}, -TΔS={TdS_mean:.2f}±{TdS_std:.2f}, ΔG={dG_gb:.2f}±{dG_gb_std:.2f}")
print(f"PB: ΔH={dH_pb:.2f}±{dH_pb_std:.2f}, -TΔS={TdS_mean:.2f}±{TdS_std:.2f}, ΔG={dG_pb:.2f}±{dG_pb_std:.2f}")

plt.rcParams.update({
# sets default font for the whole figure to Arial-like sans-serif
    'font.family': 'sans-serif',
# fallback fonts if Arial is missing
    'font.sans-serif': ['Arial', 'DejaVu Sans'],
# base font size for ticks and labels
    'font.size': 10,
# gives the light grey panel background
    'axes.facecolor': '#EAEAF2',
# removes the dark box border around the axes
    'axes.edgecolor': 'none',
# turns on the grid
    'axes.grid': True,
# makes grid lines white so they contrast with grey panel
    'grid.color': 'white',
# sets grid line thickness
    'grid.linewidth': 0.8,
# draws grid behind the bars, not on top
    'axes.axisbelow': True,
# makes x tick labels grey like the paper
    'xtick.color': '#555555',
# makes y tick labels grey like the paper
    'ytick.color': '#555555',
})
fig7_colors = {'ΔH': '#e78a9b', '-TΔS': '#5cab44', 'ΔG': '#5f9ad0'}
terms = ['ΔH', '-TΔS', 'ΔG']
colors = [fig7_colors[t] for t in terms]

def plot_fig7(vals, errs, title, save, ylabel_fs=3):
    fig, ax = plt.subplots(figsize=(4.5, 3.2))
    bars = ax.bar(terms, vals, yerr=errs, color=colors, edgecolor='white', capsize=3, width=0.6)
    #ax.bar_label(bars, fmt='%.2f', fontsize=7, label_type='center', color='black', fontweight='bold') #for values inside bars
    ax.tick_params(axis='x', labelsize=10, rotation=30, labelcolor='black') # increases x-axis label font to 14 and tilts them 25 degrees
    ax.tick_params(axis='y', labelsize=8, labelcolor='black') # increases y-axis label font to 14
    ax.xaxis.grid(False) # removes only vertical grid lines
    ax.yaxis.grid(True) # keeps horizontal grid lines
    ax.set_ylabel('Energy (kcal/mol)', fontsize=ylabel_fs)
    ax.set_title(title, fontsize=8, fontweight='bold', loc='left')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.tick_params(axis='x', length=0) # remove x-axis ticks
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(save, dpi=300)
    plt.show()

# --- GB alone ---
vals_gb = [dH_gb, TdS_mean, dG_gb]
errs_gb = [dH_gb_std, TdS_std, dG_gb_std]
plot_fig7(vals_gb, errs_gb, 'LIG_23 - GB', 'Fig3_GB_Lig_Entropy.png', ylabel_fs=9)

# --- PB alone ---
vals_pb = [dH_pb, TdS_mean, dG_pb]
errs_pb = [dH_pb_std, TdS_std, dG_pb_std]
plot_fig7(vals_pb, errs_pb, 'LIG_23 - PB', 'Fig4_PB_Lig_Entropy.png', ylabel_fs=9)

# ===== SIDE BY SIDE =====

# --- Fig6: GB vs PB side by side in SAME panel ---
def plot_fig6_side_same_format(save='Fig6_GB_PB_side_SAME_FORMAT.png'):
    plt.rcParams.update({'font.family':'sans-serif','font.size':7.5})

    fig, axes = plt.subplots(1,3, figsize=(4.5,3.2))
    fig.suptitle('lig_23 GB vs PB', fontsize=11, fontweight='bold', x=0.02, ha='left')

    colors = {'VDWAALS':'#E78CA2','EEL':'#C2A353','EGB':'#6AA84F','ESURF':'#7FC8BC',
              'EPB':'#6AA84F','ENPOLAR':'#7FC8BC','GGAS':'#4DB6AC','GSOLV':'#8A9AD5','TOTAL':'#E78CA2'}

    # Define what to compare side by side
    panel_defs = [
        ('GGAS', ['VDWAALS','EEL'], ['VDWAALS','EEL']),
        ('GSOLV', ['EGB','ESURF'], ['EPB','ENPOLAR']),
        ('TOTAL', ['GGAS','GSOLV','TOTAL'], ['GGAS','GSOLV','TOTAL'])
    ]

    x_width = 0.35
    for ax, (gtitle, keys_gb, keys_pb) in zip(axes, panel_defs):
        ax.set_facecolor(BG)
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.set_title(gtitle, fontsize=10, color='Black')
        ax.xaxis.grid(False)
        ax.yaxis.grid(True, color='white', linewidth=0.8)
        ax.set_axisbelow(True)
        ax.tick_params(axis='x', length=0)
        ax.tick_params(axis='y', labelsize=8, labelcolor='Black')

        x = np.arange(len(keys_gb))
        gb_vals = [gb[k][0] for k in keys_gb]
        gb_errs = [gb[k][1] for k in keys_gb]
        pb_vals = [pb[k][0] for k in keys_pb]
        pb_errs = [pb[k][1] for k in keys_pb]

        ax.bar(x - x_width/2, gb_vals, x_width, yerr=gb_errs, capsize=1.6,
               color=[colors[k] for k in keys_gb], edgecolor='white', label='GB')
        ax.bar(x + x_width/2, pb_vals, x_width, yerr=pb_errs, capsize=1.6,
               color=[colors[k] for k in keys_gb], edgecolor='white', alpha=0.65, hatch='///', label='PB')

        ax.axhline(0, color='black', linewidth=0.6)
        ax.set_ylim(55, -82)
        ax.set_xticks(x)
        ax.axhline(0, color='black', linewidth=0.6)
        ax.set_ylim(55, -82)
        ax.set_xticks(x)
        if gtitle == 'GSOLV':
            xlabels = ['EGB/\nEPB', 'ESURF/\nENPOLAR']
        else:
            xlabels = keys_gb
        ax.set_xticklabels(xlabels, rotation=30, ha='right', fontsize=8, color='Black')


    for ax in axes[1:]:
        ax.tick_params(axis='y', labelleft=False, left=False)

    axes[0].set_ylabel('Energy (kcal/mol)', fontsize=8)
    axes[0].legend(fontsize=6, frameon=False)
    plt.tight_layout(rect=[0,0,1,0.88])
    fig.subplots_adjust(wspace=0.04)
    plt.savefig(save, dpi=300, bbox_inches='tight')
    print(f"Saved: {save}")
    plt.show()

def plot_fig7_side_same_format(save='Fig7_GB_PB_side_SAME_FORMAT.png'):
    plt.rcParams.update({
        'font.family': 'sans-serif',
        'font.sans-serif': ['Arial', 'DejaVu Sans'],
        'font.size': 10,
        'axes.facecolor': '#EAEAF2',
        'axes.edgecolor': 'none',
        'axes.grid': True,
        'grid.color': 'white',
        'grid.linewidth': 0.8,
        'axes.axisbelow': True,
        'xtick.color': '#555555',
        'ytick.color': '#555555',
    })
    fig7_colors_local = {'ΔH': '#e78a9b', '-TΔS': '#5cab44', 'ΔG': '#5f9ad0'}
    terms_local = ['ΔH', '-TΔS', 'ΔG']
    colors_local = [fig7_colors_local[t] for t in terms_local]

    x = np.arange(len(terms_local))
    width = 0.35

    fig, ax = plt.subplots(figsize=(4.5, 3.2))
    ax.bar(x - width/2, [dH_gb, TdS_mean, dG_gb], width, yerr=[dH_gb_std, TdS_std, dG_gb_std],
           color=colors_local, edgecolor='white', capsize=2, label='GB')
    ax.bar(x + width/2, [dH_pb, TdS_mean, dG_pb], width, yerr=[dH_pb_std, TdS_std, dG_pb_std],
           color=colors_local, edgecolor='white', capsize=2, alpha=0.65, hatch='///', label='PB')

    ax.tick_params(axis='x', labelsize=10, rotation=30, labelcolor='black')
    ax.tick_params(axis='y', labelsize=8, labelcolor='black')
    ax.xaxis.grid(False)
    ax.yaxis.grid(True)
    ax.set_ylabel('Energy (kcal/mol)', fontsize=10)
    ax.set_title('LIG_23 - GB vs PB', fontsize=8, fontweight='bold', loc='left')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.tick_params(axis='x', length=0)
    ax.legend(fontsize=7, frameon=False)
    ax.invert_yaxis()
    ax.set_xticks(x)
    ax.set_xticklabels(terms_local)
    plt.tight_layout()
    plt.savefig(save, dpi=300)
    print(f"Saved: {save}")
    plt.show()

plot_fig6_side_same_format('Fig5_GB_PB_side_by_side_Energy.png')
plot_fig7_side_same_format('Fig6_GB_PB_side_SAME_entropy.png')

In [ ]:
# For TOTAL_vs_Time
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

sns.set_style("darkgrid")
plt.rcParams.update({
    'xtick.color': 'black',
    'ytick.color': 'black',
    'axes.labelcolor': 'black',
    'axes.edgecolor': 'black'
})

csv_file = 'FINAL_RESULTS_last10ns_lig_23.csv' # your uploaded file

def get_delta_df(csv_path, section):
    # section: 0 = GB Delta, 1 = PB Delta
    with open(csv_path, 'r', errors='ignore') as f:
        lines = f.readlines()
    idxs = [i for i,l in enumerate(lines) if 'Delta Energy Terms' in l]
    start = idxs[section]
    header = lines[start+1].strip().split(',')
    rows = []
    for l in lines[start+2:]:
        if 'POISSON BOLTZMANN:' in l or 'GENERALIZED BORN:' in l:
            break
        if not l.strip() or 'Frame #' in l:
            continue
        p = l.strip().split(',')
        if len(p)!= len(header):
            continue
        try:
            rows.append([int(p[0])] + [float(x) for x in p[1:]])
        except:
            continue
    df = pd.DataFrame(rows, columns=header)
    return df

# THIS creates df_gb and df_pb - this was missing
df_gb = get_delta_df(csv_file, 0)
df_pb = get_delta_df(csv_file, 1)

print("df_gb:", df_gb.shape, "frames", df_gb['Frame #'].min(), "-", df_gb['Frame #'].max())
print("df_pb:", df_pb.shape, "frames", df_pb['Frame #'].min(), "-", df_pb['Frame #'].max())

def plot_total(df, title, fname):
    df_plot = df[(df['Frame #']>=4001) & (df['Frame #']<=5001)].copy()
    df_plot['rel_frame'] = df_plot['Frame #'] - 4001

    fig, ax = plt.subplots(figsize=(5,2.5))
    ax.plot(df_plot['rel_frame'], df_plot['TOTAL'], color='#6e6e6e', linewidth=0.6, label='TOTAL')
    # --- article version, starts a bit later ---
    ax.plot(df_plot['rel_frame'], df_plot['TOTAL'].rolling(50, min_periods=1).mean(),
            color='red', linewidth=0.8, linestyle='--', label='Mov. Av.')

    ax.set_xlim(-35, 1020) # little space before 0 like article
    ax.set_xticks([0, 200, 400, 600, 800, 1000])
    ax.set_xticklabels([4001, 4200, 4400, 4600, 4800, 5001]) # Show 0-1000 as 4001-5001 on x-axis
    ax.tick_params(axis='x', labelsize=7)
    ax.tick_params(axis='y', labelsize=7)
    ax.set_xlabel('Frames', fontsize=8, color='black')
    ax.set_ylabel('Energy (kcal/mol)', fontsize=8, color='black')
    ax.set_title(title, fontsize=9, fontweight='bold', loc='left')
    ax.legend(fontsize=6, loc='lower right')
    sns.despine(left=True, bottom=True)
    plt.tight_layout()
    plt.savefig(fname, dpi=300, bbox_inches='tight')
    plt.show()

# then call as before:
plot_total(df_gb, 'LIG_23 - GB', 'Fig7_TOTAL_vs_Time_GB_4001_5001.png')
plot_total(df_pb, 'LIG_23 - PB', 'Fig8_TOTAL_vs_Time_PB_4001_5001.png')

# for pb vs gb
def plot_total_overlay(df_gb, df_pb, save='Fig9_TOTAL_vs_Time_GB_PB_overlay.png'):
    df_gb_plot = df_gb[(df_gb['Frame #']>=4001) & (df_gb['Frame #']<=5001)].copy()
    df_pb_plot = df_pb[(df_pb['Frame #']>=4001) & (df_pb['Frame #']<=5001)].copy()
    df_gb_plot['rel_frame'] = df_gb_plot['Frame #'] - 4001
    df_pb_plot['rel_frame'] = df_pb_plot['Frame #'] - 4001

    fig, ax = plt.subplots(figsize=(5, 2.5))
    ax.plot(df_gb_plot['rel_frame'], df_gb_plot['TOTAL'], color='#5BBAB5', linewidth=0.6, label='GB TOTAL')
    ax.plot(df_gb_plot['rel_frame'], df_gb_plot['TOTAL'].rolling(50, min_periods=1).mean(), color='#0B2545', linewidth=1.0, linestyle='--')
    ax.plot(df_pb_plot['rel_frame'], df_pb_plot['TOTAL'], color='#E8AFAF', linewidth=0.6, label='PB TOTAL')
    ax.plot(df_pb_plot['rel_frame'], df_pb_plot['TOTAL'].rolling(50, min_periods=1).mean(), color='red', linewidth=1.0, linestyle='--')

    ax.set_xlim(-35, 1020)
    ax.set_xticks([0, 200, 400, 600, 800, 1000])
    ax.set_xticklabels([4001, 4200, 4400, 4600, 4800, 5001])
    ax.tick_params(axis='x', labelsize=7)
    ax.tick_params(axis='y', labelsize=7)
    ax.set_xlabel('Frames', fontsize=8, color='black')
    ax.set_ylabel('Energy (kcal/mol)', fontsize=8, color='black')
    ax.set_title('LIG_23 | GB vs PB | TOTAL', fontsize=9, fontweight='bold', loc='left')
    ax.legend(fontsize=6, loc='lower right')
    sns.despine(left=True, bottom=True)
    plt.tight_layout()
    plt.savefig(save, dpi=300, bbox_inches='tight')
    plt.show()

plot_total_overlay(df_gb, df_pb)

In [ ]:
# Run must after cell 2 for rest figures__code to find top10 residues
filename = 'FINAL_DECOMP_last10ns_lig_23.dat'

with open(filename, 'r') as f:
    lines = f.readlines()

deltas_indices = []
for i, l in enumerate(lines):
    if l.strip() == 'DELTAS:' and i+1 < len(lines) and 'Total Energy Decomposition' in lines[i+1]:
        deltas_indices.append(i)

gb_start = deltas_indices[0] + 3
pb_start = deltas_indices[1] + 3

def parse_top10_with_err(start):
    data = []
    for l in lines[start:]:
        s = l.strip()
        if not s or 'Sidechain Energy' in s or 'Backbone Energy' in s or s.startswith('|'):
            break
        if s.startswith('Residue') or s.startswith(',Avg'):
            continue
        parts = s.split(',')
        if len(parts) < 19:
            continue
        res = parts[0]
        if not (res.startswith('R:A:') or res.startswith('L:B:')):
            continue
        try:
            avg = float(parts[16])
            std = float(parts[17])
            serr = float(parts[18])
            data.append((res, avg, std, serr))
        except:
            pass
    return sorted(data, key=lambda x: x[1])[:10]

top10_gb = parse_top10_with_err(gb_start)
top10_pb = parse_top10_with_err(pb_start)

print("=== Top 10 GB DELTAS with error ===")
for r, avg, std, serr in top10_gb:
    print(f"{r:15s} {avg:8.3f} ± {std:6.3f} (SD), ± {serr:6.3f} (SE)")

print("\n=== Top 10 PB DELTAS with error ===")
for r, avg, std, serr in top10_pb:
    print(f"{r:15s} {avg:8.3f} ± {std:6.3f} (SD), ± {serr:6.3f} (SE)")

# For plotting with error bars later:
# gb_residues = [x[0] for x in top10_gb]
# gb_means = [x[1] for x in top10_gb]
# gb_stds = [x[2] for x in top10_gb]

In [ ]:
# for per residue bar charts without ligand
def parse_top10_noLig(start):
    data = []
    for l in lines[start:]:
        s = l.strip()
        if not s or 'Sidechain Energy' in s or 'Backbone Energy' in s or s.startswith('|'):
            break
        if s.startswith('Residue') or s.startswith(',Avg'):
            continue
        parts = s.split(',')
        if len(parts) < 19:
            continue
        res = parts[0]
        if res.startswith('L:B:'):
            continue
        if not res.startswith('R:A:'):
            continue
        try:
            avg = float(parts[16]); std = float(parts[17]); serr = float(parts[18])
            data.append((res, avg, std, serr))
        except:
            pass
    return sorted(data, key=lambda x: x[1])[:10]

top10_gb = parse_top10_noLig(gb_start)
top10_pb = parse_top10_noLig(pb_start)

print("GB without ligand:")
for r, avg, std, serr in top10_gb:
    print(f"{r:15s} {avg:8.3f} ± {std:.3f}")
print("\nPB without ligand:")
for r, avg, std, serr in top10_pb:
    print(f"{r:15s} {avg:8.3f} ± {std:.3f}")

#----per residue bar chart-----
import matplotlib.pyplot as plt
import numpy as np

fig8_colors = ['#E8AFAF','#DDA16E','#A3A33B','#7FC97F','#8ED6A0',
               '#5BBAB5','#6BAED6','#9E9AC8','#C39BD3','#F194B8']

def plot_per_residue(top10_data, title, save):
    res = [x[0].replace('R:A:','') for x in top10_data]
    mean_vals = np.array([x[1] for x in top10_data])

    fig, ax = plt.subplots(figsize=(4.5, 3.2))
    ax.set_facecolor('#EAEAF2')

    bars = ax.bar(res, mean_vals,
           color=fig8_colors[:len(res)],
           edgecolor='none', width=0.7, zorder=3)

    ax.set_ylabel('Energy (kcal/mol)', fontsize=8, color='black')
    ax.set_title(title, fontsize=6.8, color='black', fontweight='bold', loc='left', pad=12)
    ax.tick_params(axis='x', labelsize=6.8, labelcolor='black', which='both', bottom=False, top=False)
    ax.tick_params(axis='y', labelsize=7, labelcolor='black')
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8.2, color='black')

    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.invert_yaxis()

    ax.set_ylim(0.0, mean_vals.min() - 0.29)

    # value label at top of bar, centered
    ax.bar_label(bars, fmt='%.2f', fontsize=6.6, color='black', padding=0.5, label_type='edge')

    ax.yaxis.grid(True, color='white', linewidth=0.8, zorder=0)
    ax.xaxis.grid(False)
    ax.set_axisbelow(True)
    plt.tight_layout()
    plt.savefig(save, dpi=300, bbox_inches='tight')
    plt.show()

plot_per_residue(top10_gb, 'A. LIG_23 | GB | DELTAS', 'fig10_GB_dat_top10_res.png')
plot_per_residue(top10_pb, 'B. LIG_23 | PB | DELTAS', 'fig11_PB_dat_top10_res.png')

# ===== SIDE BY SIDE - PER RESIDUE =====
def parse_all_noLig(start):
    data = {}
    for l in lines[start:]:
        s = l.strip()
        if not s or 'Sidechain Energy' in s or 'Backbone Energy' in s or s.startswith('|'):
            break
        if s.startswith('Residue') or s.startswith(',Avg'):
            continue
        parts = s.split(',')
        if len(parts) < 19:
            continue
        res = parts[0]
        if res.startswith('L:B:'):
            continue
        if not res.startswith('R:A:'):
            continue
        try:
            avg = float(parts[16])
            data[res] = avg
        except:
            pass
    return data

gb_dict = parse_all_noLig(gb_start)
pb_dict = parse_all_noLig(pb_start)

# Take union of your top10s so we compare same residues side by side
# keeps your order from GB top10
residues_for_side = [x[0] for x in top10_gb]  # e.g. ARG_132, TRP_117...
# if PB has residues not in GB, add them
for r,_,_,_ in top10_pb:
    if r not in residues_for_side:
        residues_for_side.append(r)
residues_for_side = residues_for_side[:10]  # keep 10

gb_vals_side = [gb_dict.get(r, 0) for r in residues_for_side]
pb_vals_side = [pb_dict.get(r, 0) for r in residues_for_side]
res_labels = [r.replace('R:A:','') for r in residues_for_side]

fig8_colors_side = ['#E8AFAF','#DDA16E','#A3A33B','#7FC97F','#8ED6A0',
               '#5BBAB5','#6BAED6','#9E9AC8','#C39BD3','#F194B8']

def plot_per_residue_side_same_format(save='LIG_23_GB_PB_perRes_side_SAME.png'):
    fig, ax = plt.subplots(figsize=(4.5, 3.2))
    ax.set_facecolor('#EAEAF2')

    x = np.arange(len(residues_for_side))
    width = 0.35

    ax.bar(x - width/2, gb_vals_side, width,
           color=fig8_colors_side[:len(res_labels)],
           edgecolor='white', zorder=3, label='GB')
    ax.bar(x + width/2, pb_vals_side, width,
           color=fig8_colors_side[:len(res_labels)],
           edgecolor='white', alpha=0.65, hatch='///', zorder=3, label='PB')

    ax.set_ylabel('Energy (kcal/mol)', fontsize=8, color='black')
    ax.set_title('LIG_23 | GB vs PB | DELTAS', fontsize=6.8, color='black', fontweight='bold', loc='left', pad=12)
    ax.tick_params(axis='x', labelsize=6.8, labelcolor='black', which='both', bottom=False, top=False)
    ax.tick_params(axis='y', labelsize=7, labelcolor='black')
    ax.set_xticks(x)
    ax.set_xticklabels(res_labels, rotation=45, ha='right', fontsize=8.2, color='black')

    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.invert_yaxis()
    min_val = min(min(gb_vals_side), min(pb_vals_side))
    ax.set_ylim(0.0, min_val - 0.5)

    ax.invert_yaxis()
    all_vals = gb_vals_side + pb_vals_side
    min_val = min(all_vals)
    max_val = max(all_vals)
    bottom = max(0.5, max_val + 0.5) if max_val > 0 else 0.0
    top = min_val - 0.5
    ax.set_ylim(bottom, top)
    ax.axhline(0, color='black', linewidth=0.6, zorder=4)

    # USE containers for stored values: ax.containers[0] = first bar group (GB), ax.containers[1] = second (PB).

    #ax.bar_label(ax.containers[0], fmt='%.2f', fontsize=5.5, color='black', padding=1, label_type='edge')
    #ax.bar_label(ax.containers[1], fmt='%.2f', fontsize=5.5, color='black', padding=1, label_type='edge')

    ax.yaxis.grid(True, color='white', linewidth=0.8, zorder=0)
    ax.xaxis.grid(False)
    ax.set_axisbelow(True)
    ax.legend(fontsize=6, frameon=False, handlelength=2, handleheight=1, borderpad=0.4, handletextpad=0.5)
    plt.tight_layout()
    plt.savefig(save, dpi=300, bbox_inches='tight')
    print(f"Saved: {save}")
    plt.show()

plot_per_residue_side_same_format('Fig12_GB_based_PB_perRes_side_by_side.png')

In [ ]:
# Full GB only & GB vs GB based PB heatmap
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

filename = 'FINAL_DECOMP_last10ns_lig_23.csv'

with open(filename, 'r') as f:
    lines = f.readlines()

# Find DELTAS blocks dynamically
deltas_indices = [i for i,l in enumerate(lines) if l.strip() == "DELTAS:"]
print("DELTAS at:", deltas_indices)
# [162187, 405464] expected -> 0 = GB, 1 = PB
gb_delta_start = deltas_indices[0] + 1

pb_start = next(i for i,l in enumerate(lines) if "Poisson Boltzmann" in l)
print("PB starts at:", pb_start)
gb_delta_end = pb_start

wanted = ['R:A:ARG:263','R:A:THR:266','R:A:MET:231','R:A:MET:250',
          'R:A:VAL:253','R:A:PHE:254','R:A:PHE:270','R:A:LEU:267',
          'R:A:PHE:228','R:A:VAL:249','L:B:LIG:327']

data = []
for l in lines[gb_delta_start:gb_delta_end]:
    ls = l.strip()
    if not ls or ls.startswith('"') or "Frame" in ls:
        continue
    if "DELTAS" in ls or "Generalized" in ls or "Poisson" in ls:
        continue
    parts = [p.strip() for p in ls.split(',')]
    if len(parts) >= 8 and parts[0].isdigit() and parts[1] in wanted:
        data.append(parts)

print(f"Parsed rows raw: {len(data)}")

df = pd.DataFrame(data, columns=['Frame','Residue','Internal','vdW','Elec','Pol','NonPol','TOTAL'])
df['TOTAL'] = pd.to_numeric(df['TOTAL'], errors='coerce')
df['Frame'] = pd.to_numeric(df['Frame'], errors='coerce')
df = df.dropna(subset=['TOTAL','Frame'])

before = len(df)
df = df.drop_duplicates(subset=['Frame','Residue'])
print(f"After dedup: {len(df)} (removed {before-len(df)})")
print(f"Unique frames: {df['Frame'].nunique()}, min {int(df['Frame'].min())} max {int(df['Frame'].max())}")

mean_gb = df.groupby('Residue')['TOTAL'].mean().reindex(wanted)
print("\nMean GB (kcal/mol):\n", mean_gb.round(3))

pivot = df.pivot_table(index='Residue', columns='Frame', values='TOTAL', aggfunc='mean').reindex(wanted)
pivot = pivot.reindex(sorted(pivot.columns), axis=1)

pivot_gb = pivot

# Parse PB
pb_delta_start = deltas_indices[1] + 3
data_pb = []
for l in lines[pb_delta_start:]:
    ls = l.strip()
    if not ls or ls.startswith('"') or "Frame" in ls:
        continue
    if "DELTAS" in ls or "Poisson" in ls or "Generalized" in ls:
        break
    parts = [p.strip() for p in ls.split(',')]
    if len(parts) >= 8 and parts[0].isdigit() and parts[1] in wanted:
        data_pb.append(parts)

df_pb = pd.DataFrame(data_pb, columns=['Frame','Residue','Internal','vdW','Elec','Pol','NonPol','TOTAL'])
df_pb['TOTAL'] = pd.to_numeric(df_pb['TOTAL'], errors='coerce')
df_pb['Frame'] = pd.to_numeric(df_pb['Frame'], errors='coerce')
df_pb = df_pb.dropna(subset=['TOTAL','Frame']).drop_duplicates(subset=['Frame','Residue'])
pivot_pb = df_pb.pivot_table(index='Residue', columns='Frame', values='TOTAL', aggfunc='mean').reindex(wanted)
pivot_pb = pivot_pb.reindex(sorted(pivot_pb.columns), axis=1)

# 1. HEATMAP
fig, ax = plt.subplots(figsize=(12,5), dpi=300)
im = ax.imshow(pivot.values, aspect='auto', cmap='RdBu_r', vmin=-3.5, vmax=1.5, interpolation='nearest')
cbar = plt.colorbar(im, ax=ax, label='ΔG (kcal/mol)')
cbar.set_ticks([-3, -2, -1, 0, 1])
ax.set_yticks(range(len(wanted)))
ax.set_yticklabels([r.replace('R:A:','A:').replace('L:B:','B:') for r in wanted], fontsize=12.2, color='black')
ax.yaxis.tick_right()
ax.yaxis.set_label_position("right")

ax.set_xticks(np.linspace(0, pivot.shape[1]-1, 5))
ax.set_xticklabels([int(pivot.columns[int(i)]) for i in np.linspace(0, pivot.shape[1]-1, 5)],fontsize=11, color='black')

#ax.set_xlim(0, pivot.shape[1]-1) #this over write it
#ax.set_xticks([0, 100, 200, 300, 400, 500, 600, 700, 800, 900])
#ax.set_xticklabels([1, 101, 201, 301, 401, 501, 601, 701, 801, 901])

ax.set_xlabel('Frames', fontsize=12)
ax.set_title('Per-Residue Heatmap | MM/GBSA | DELTAS', fontweight='bold', pad=20)
ax.grid(False)

row_types = np.array([0 if 'LIG' in r else 1 for r in wanted])[:, None]
cmap_ann = ListedColormap(['#00FF00', '#E94E77'])
ax_pos = ax.get_position()
ax_ann = fig.add_axes([ax_pos.x0 - 0.008, ax_pos.y0, 0.007, ax_pos.y1 - ax_pos.y0])
ax_ann.imshow(row_types, aspect='auto', cmap=cmap_ann, vmin=0, vmax=1, interpolation='nearest')
ax_ann.set_xticks([]); ax_ann.set_yticks([])

for spine in ax_ann.spines.values():
    spine.set_visible(False)
    spine.set_edgecolor('black')
    spine.set_linewidth(1.0)

for spine in ax.spines.values():
    spine.set_visible(False)

cbar.outline.set_visible(True)
cbar.outline.set_edgecolor('black')
cbar.outline.set_linewidth(0.9)
cbar.ax.set_position([ax_pos.x0 - 0.029, ax_pos.y0, 0.01, ax_pos.y1 - ax_pos.y0])
cbar.ax.yaxis.set_ticks_position('left')
cbar.ax.yaxis.set_label_position('left')
cbar.ax.tick_params(colors='black', labelsize=9.5)
plt.setp(cbar.ax.get_yticklabels(), color='black')
cbar.set_label('ΔG (kcal/mol)', labelpad=5,fontsize=12)

legend_handles = [Patch(facecolor='#E94E77', edgecolor='none', label='Receptor'),
                  Patch(facecolor='#00FF00', edgecolor='none', label='Ligand')]
ax.legend(handles=legend_handles, loc='lower left', bbox_to_anchor=(0.35, 0.98),
          ncol=2, frameon=False, fontsize=10)
plt.savefig('Fig13_heatmap_GB.png', dpi=600, bbox_inches='tight')
plt.show()

# 2. BARCHART
wanted_no_lig = [r for r in wanted if 'LIG' not in r]
mean_gb_no_lig = mean_gb.reindex(wanted_no_lig)
x = np.arange(len(wanted_no_lig))
plt.figure(figsize=(12,5), dpi=300)
plt.bar(x, mean_gb_no_lig.values, color='#FFC000', edgecolor='black')
plt.axhline(0, color='black', linewidth=1)
plt.xticks(x, [r.replace('R:A:','').replace('L:B:','') for r in wanted_no_lig], rotation=45, ha='right')
plt.ylabel('DELTA TOTAL (kcal/mol)', fontweight='bold')
plt.title('Per-Residue Decomposition | MM/GBSA | DELTAS', fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.gca().xaxis.grid(False)
plt.tight_layout()
plt.savefig('barchart_GB_ONLY_NoLig.png', dpi=600)
plt.show()

# ========== ONE PANEL HEATMAP - GB top / PB bottom ==========

fig, ax = plt.subplots(figsize=(12, 5), dpi=300)

#pivot_gb = pivot
# stack: GB + 1 white row + PB
combined = np.vstack([pivot_gb.values, np.full((1, pivot_gb.shape[1]), np.nan), pivot_pb.values])

cmap = plt.get_cmap('RdBu_r').copy()
cmap.set_bad('white')
im = ax.imshow(combined, aspect='auto', cmap=cmap, vmin=-3.5, vmax=1.5, interpolation='nearest')
cbar = plt.colorbar(im, ax=ax, label='ΔG (kcal/mol)')
cbar.set_ticks([-3, -2, -1, 0, 1])

y_labels = [r.replace('R:A:','A:').replace('L:B:','B:') for r in wanted] + [''] + [r.replace('R:A:','A:').replace('L:B:','B:') for r in wanted]
ax.set_yticks(range(len(y_labels)))
ax.set_yticklabels(y_labels, fontsize=12.2, color='black')
ax.yaxis.tick_right()
ax.yaxis.set_label_position("right")

ax.set_xticks(np.linspace(0, pivot_gb.shape[1]-1, 5))
ax.set_xticklabels([int(pivot_gb.columns[int(i)]) for i in np.linspace(0, pivot_gb.shape[1]-1, 5)], fontsize=11, color='black')

ax.set_xlabel('Frames', fontsize=12)
ax.set_title('Per-Residue Heatmap | MM/GBSA vs MM/PBSA | DELTAS', fontweight='bold', pad=24)
ax.grid(False)

ax.axhline(len(wanted)-0.5, color='black', linewidth=0)
ax.axhline(len(wanted)+0.5, color='black', linewidth=0)

row_types_single = np.array([0 if 'LIG' in r else 1 for r in wanted])
row_types_combined = np.vstack([row_types_single[:, None], np.array([[2]]), row_types_single[:, None]]) # 2 = white gap
cmap_ann = ListedColormap(['#00FF00', '#E94E77', 'white'])
ax_pos = ax.get_position()
ax_ann = fig.add_axes([ax_pos.x0 - 0.008, ax_pos.y0, 0.007, ax_pos.y1 - ax_pos.y0])
ax_ann.imshow(row_types_combined, aspect='auto', cmap=cmap_ann, vmin=0, vmax=2, interpolation='nearest')
ax_ann.set_xticks([]); ax_ann.set_yticks([])
for spine in ax_ann.spines.values():
    spine.set_visible(False)

for spine in ax.spines.values():
    spine.set_visible(False)

cbar.outline.set_visible(True)
cbar.outline.set_edgecolor('black')
cbar.outline.set_linewidth(0.9)
cbar.ax.set_position([ax_pos.x0 - 0.029, ax_pos.y0, 0.01, ax_pos.y1 - ax_pos.y0])
cbar.ax.yaxis.set_ticks_position('left')
cbar.ax.yaxis.set_label_position('left')
cbar.ax.tick_params(colors='black', labelsize=9.5)
plt.setp(cbar.ax.get_yticklabels(), color='black')
cbar.set_label('ΔG (kcal/mol)', labelpad=5, fontsize=12)

# GB / PB
ax.text(-0.019, 0.88, 'GB', transform=ax.transAxes, fontsize=9, fontweight='bold', va='center', ha='right')
ax.text(-0.019, 0.12, 'PB', transform=ax.transAxes, fontsize=9, fontweight='bold', va='center', ha='right')

legend_handles = [Patch(facecolor='#E94E77', edgecolor='none', label='Receptor'),
                  Patch(facecolor='#00FF00', edgecolor='none', label='Ligand')]
ax.legend(handles=legend_handles, loc='lower left', bbox_to_anchor=(0.35, 1.01),
          ncol=2, frameon=False, fontsize=10)

plt.savefig('Fig_14_heatmap_GB_&_GB_based_PB_ONE_PANEL.png', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
# PB heatmap for your cherry-picked residues
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

filename = 'FINAL_DECOMP_last10ns_lig_23.csv'

with open(filename, 'r') as f:
    lines = f.readlines()

deltas_idx = [i for i,l in enumerate(lines) if l.strip() == "DELTAS:"]
pb_delta_start = deltas_idx[1] + 3  # +1=TDC title, +2=header, +3=first data row (was +1 before)
print("PB DELTAS at:", deltas_idx[1], "-> data from:", pb_delta_start)

# your cherry-picked residues - unchanged
wanted = ['R:A:ARG:263','R:A:THR:266','R:A:MET:231','R:A:MET:250','R:A:VAL:253','R:A:PHE:254','R:A:PHE:270','R:A:LEU:267','R:A:PHE:228','R:A:VAL:249','L:B:LIG:327']

data=[]
for l in lines[pb_delta_start:]:
    ls=l.strip()
    if not ls or "Frame" in ls or ls.startswith('"') or "DELTAS" in ls or "Poisson" in ls or "Generalized" in ls:
        continue
    parts=[p.strip() for p in ls.split(',')]
    if len(parts)>=8 and parts[0].isdigit() and parts[1] in wanted:
        data.append(parts)

print(f"Parsed rows raw: {len(data)}")

df = pd.DataFrame(data, columns=['Frame','Residue','Internal','vdW','Elec','Pol','NonPol','TOTAL'])
df['TOTAL']=pd.to_numeric(df['TOTAL'], errors='coerce')
df['Frame']=pd.to_numeric(df['Frame'], errors='coerce')
df = df.dropna(subset=['TOTAL','Frame'])

before = len(df)
df = df.drop_duplicates(subset=['Frame','Residue'])
print(f"After dedup: {len(df)} (removed {before-len(df)})")
print(f"Unique frames: {df['Frame'].nunique()}, min {int(df['Frame'].min())} max {int(df['Frame'].max())}")

mean_pb = df.groupby('Residue')['TOTAL'].mean().reindex(wanted)
print("\nMean PB (kcal/mol):\n", mean_pb.round(4))

pivot = df.pivot_table(index='Residue', columns='Frame', values='TOTAL', aggfunc='mean').reindex(wanted)
pivot = pivot.reindex(sorted(pivot.columns), axis=1)

fig, ax = plt.subplots(figsize=(12,5), dpi=300)
im = ax.imshow(pivot.values, aspect='auto', cmap='RdBu_r', vmin=-3.5, vmax=1.5, interpolation='nearest')
cbar = plt.colorbar(im, ax=ax, label='ΔG (kcal/mol)')
cbar.set_ticks([-3, -2, -1, 0, 1])
ax.set_yticks(range(len(wanted)))
ax.yaxis.tick_right()
ax.yaxis.set_label_position("right")
ax.set_yticklabels([r.replace('R:A:','A:').replace('L:B:','B:') for r in wanted], fontsize=12.2, color='black')
ax.set_xticks(np.linspace(0, pivot.shape[1]-1, 5))
ax.set_xticklabels([int(pivot.columns[int(i)]) for i in np.linspace(0, pivot.shape[1]-1, 5)], fontsize=11, color='black')
ax.set_xlabel('Frames', fontsize=12)
ax.set_title('Per-Residue Heatmap | MM/PBSA | DELTA | Lig23', fontweight='bold', pad=20)
ax.grid(False)

row_types = np.array([0 if 'LIG' in r else 1 for r in wanted])[:, None]
cmap_ann = ListedColormap(['#00FF00', '#E94E77'])
ax_pos = ax.get_position()
ax_ann = fig.add_axes([ax_pos.x0 - 0.008, ax_pos.y0, 0.007, ax_pos.y1 - ax_pos.y0])
ax_ann.imshow(row_types, aspect='auto', cmap=cmap_ann, vmin=0, vmax=1, interpolation='nearest')
ax_ann.set_xticks([]); ax_ann.set_yticks([])

for spine in ax_ann.spines.values():
    spine.set_visible(False)
    spine.set_edgecolor('black')
    spine.set_linewidth(1.0)

for spine in ax.spines.values():
    spine.set_visible(False)

cbar.outline.set_visible(True)
cbar.outline.set_edgecolor('black')
cbar.outline.set_linewidth(0.9)
cbar.ax.set_position([ax_pos.x0 - 0.029, ax_pos.y0, 0.01, ax_pos.y1 - ax_pos.y0])
cbar.ax.yaxis.set_ticks_position('left')
cbar.ax.yaxis.set_label_position('left')
cbar.ax.tick_params(colors='black', labelsize=9.5)
plt.setp(cbar.ax.get_yticklabels(), color='black')
cbar.set_label('ΔG (kcal/mol)', labelpad=5,fontsize=12)

legend_handles = [Patch(facecolor='#E94E77', edgecolor='none', label='Receptor'),
                  Patch(facecolor='#00FF00', edgecolor='none', label='Ligand')]
ax.legend(handles=legend_handles, loc='lower left', bbox_to_anchor=(0.35, 0.98),
          ncol=2, frameon=False, fontsize=10)
plt.savefig('Fig15_heatmap_PB_cherry_picked_res.png', dpi=600, bbox_inches='tight')
plt.show()

wanted_no_lig = [r for r in wanted if 'LIG' not in r]
mean_pb_no_lig = mean_pb.reindex(wanted_no_lig)
x = np.arange(len(wanted_no_lig))
plt.figure(figsize=(12,5), dpi=300)
plt.bar(x, mean_pb_no_lig.values, color='#FFC000', edgecolor='black')
plt.axhline(0,color='black')
plt.xticks(x, [r.replace('R:A:','').replace('L:B:','') for r in wanted_no_lig], rotation=45, ha='right')
plt.ylabel('DELTA TOTAL (kcal/mol)', fontweight='bold')
plt.title('Per-Residue Decomposition | MM/PBSA | DELTA | Lig23', fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.gca().xaxis.grid(False)
plt.tight_layout()
plt.savefig('barchart_PB_ONLY_noLig.png', dpi=600)
plt.show()

In [ ]:
# Full PB heatmap for Top 10 residues
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

filename = 'FINAL_DECOMP_last10ns_lig_23.csv' #

with open(filename, 'r') as f:
    lines = f.readlines()

# Find DELTAS blocks dynamically
deltas_indices = [i for i, l in enumerate(lines) if l.strip() == "DELTAS:"]
print("DELTAS at (0-based):", deltas_indices)
# [162187, 405464] -> 0 = GB, 1 = PB

# PB block: DELTAS: at 0-based 405464, +1 = TDC title, +2 = header, +3 = first data row
pb_delta_start = deltas_indices[1] + 3
pb_delta_end = len(lines)
print("PB data from (0-based):", pb_delta_start, "to", pb_delta_end)

wanted = ['R:A:MET:250', 'R:A:LEU:267', 'R:A:THR:266', 'R:A:PHE:254',
          'R:A:MET:231', 'R:A:VAL:253', 'R:A:PHE:270', 'R:A:PHE:228',
          'R:A:VAL:258', 'R:A:LEU:246', 'L:B:LIG:327']

data = []
for l in lines[pb_delta_start:pb_delta_end]:
    ls = l.strip()
    if not ls or ls.startswith('"') or "Frame" in ls:
        continue
    if "DELTAS" in ls or "Generalized" in ls or "Poisson" in ls:
        continue
    parts = [p.strip() for p in ls.split(',')]
    if len(parts) >= 8 and parts[0].isdigit() and parts[1] in wanted:
        data.append(parts)

print(f"Parsed rows raw: {len(data)}")

df = pd.DataFrame(data, columns=['Frame', 'Residue', 'Internal', 'vdW', 'Elec', 'Pol', 'NonPol', 'TOTAL'])
df['TOTAL'] = pd.to_numeric(df['TOTAL'], errors='coerce')
df['Frame'] = pd.to_numeric(df['Frame'], errors='coerce')
df = df.dropna(subset=['TOTAL', 'Frame'])

before = len(df)
df = df.drop_duplicates(subset=['Frame', 'Residue'])
print(f"After dedup: {len(df)} (removed {before - len(df)})")
print(f"Unique frames: {df['Frame'].nunique()}, min {int(df['Frame'].min())} max {int(df['Frame'].max())}")

mean_pb = df.groupby('Residue')['TOTAL'].mean().reindex(wanted)
print("\nMean PB (kcal/mol):\n", mean_pb.round(4))

pivot = df.pivot_table(index='Residue', columns='Frame', values='TOTAL', aggfunc='mean').reindex(wanted)
pivot = pivot.reindex(sorted(pivot.columns), axis=1)

# 1. HEATMAP
fig, ax = plt.subplots(figsize=(12, 5), dpi=300)
im = ax.imshow(pivot.values, aspect='auto', cmap='RdBu_r', vmin=-3.5, vmax=1.5, interpolation='nearest')
cbar = plt.colorbar(im, ax=ax, label='ΔG (kcal/mol)')
cbar.set_ticks([-3, -2, -1, 0, 1])
ax.set_yticks(range(len(wanted)))
ax.set_yticklabels([r.replace('R:A:', 'A:').replace('L:B:', 'B:') for r in wanted], fontsize=12.2, color='black')
ax.yaxis.tick_right()
ax.yaxis.set_label_position("right")

ax.set_xticks(np.linspace(0, pivot.shape[1] - 1, 5))
ax.set_xticklabels([int(pivot.columns[int(i)]) for i in np.linspace(0, pivot.shape[1] - 1, 5)],fontsize=11, color='black')

ax.set_xlabel('Frames', fontsize=12)
ax.set_title('Per-Residue Heatmap | MM/PBSA | DELTAS', fontweight='bold', pad=20)
ax.grid(False)

row_types = np.array([0 if 'LIG' in r else 1 for r in wanted])[:, None]
cmap_ann = ListedColormap(['#00FF00', '#E94E77'])
ax_pos = ax.get_position()
ax_ann = fig.add_axes([ax_pos.x0 - 0.008, ax_pos.y0, 0.007, ax_pos.y1 - ax_pos.y0])
ax_ann.imshow(row_types, aspect='auto', cmap=cmap_ann, vmin=0, vmax=1, interpolation='nearest')
ax_ann.set_xticks([])
ax_ann.set_yticks([])
for spine in ax_ann.spines.values():
    spine.set_visible(False)
    spine.set_edgecolor('black')
    spine.set_linewidth(1.0)

for spine in ax.spines.values():
    spine.set_visible(False)

cbar.outline.set_visible(True)
cbar.outline.set_edgecolor('black')
cbar.outline.set_linewidth(0.9)
cbar.ax.set_position([ax_pos.x0 - 0.029, ax_pos.y0, 0.01, ax_pos.y1 - ax_pos.y0])
cbar.ax.yaxis.set_ticks_position('left')
cbar.ax.yaxis.set_label_position('left')
cbar.ax.tick_params(colors='black', labelsize=9.5)
plt.setp(cbar.ax.get_yticklabels(), color='black')
cbar.set_label('ΔG (kcal/mol)', labelpad=5, fontsize=12)

legend_handles = [Patch(facecolor='#E94E77', edgecolor='none', label='Receptor'),
                  Patch(facecolor='#00FF00', edgecolor='none', label='Ligand')]
ax.legend(handles=legend_handles, loc='lower left', bbox_to_anchor=(0.35, 0.98),
          ncol=2, frameon=False, fontsize=10)
plt.savefig('Fig16_heatmap_PB_Top10_res.png', dpi=600, bbox_inches='tight')
plt.show()

# 2. BARCHART
wanted_no_lig = [r for r in wanted if 'LIG' not in r]
mean_pb_no_lig = mean_pb.reindex(wanted_no_lig)
x = np.arange(len(wanted_no_lig))
plt.figure(figsize=(12, 5), dpi=300)
plt.bar(x, mean_pb_no_lig.values, color='#FFC000', edgecolor='black')
plt.axhline(0, color='black', linewidth=1)
plt.xticks(x, [r.replace('R:A:', '').replace('L:B:', '') for r in wanted_no_lig], rotation=45, ha='right')
plt.ylabel('DELTA TOTAL (kcal/mol)', fontweight='bold')
plt.title('Per-Residue Decomposition | MM/PBSA | DELTAS', fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.gca().xaxis.grid(False)
plt.tight_layout()
plt.savefig('barchart_PB_ONLY_noLig.png', dpi=600)
plt.show()